In [2]:
import boto3
import os
from dotenv import load_dotenv

In [18]:
load_dotenv()
s3 = boto3.client('s3')
ec2 = boto3.client('ec2', region_name=os.getenv('AWS_REGION'))
efs_client = boto3.client('efs', region_name=os.getenv('AWS_REGION'))
print("Sesión iniciada correctamente.")

Sesión iniciada correctamente.


In [22]:


load_dotenv()

ec2_resource = boto3.resource('ec2', region_name=os.getenv('AWS_REGION'))
ec2_client = boto3.client('ec2', region_name=os.getenv('AWS_REGION'))

def manage_ec2_improved():

    user_data_script = """#!/bin/bash
    echo "Archivo creado durante la inicialización" > /home/ec2-user/ejercicio.txt
    """

    print("Creando instancia...")
    instances = ec2_resource.create_instances(
        ImageId='ami-02dfbd4ff395f2a1b', 
        InstanceType='t2.micro',
        MinCount=1,
        MaxCount=1,
        UserData=user_data_script 
    )
    instance = instances[0]
    instance.wait_until_running()
    print(f"Instancia {instance.id} en marcha en la zona {instance.placement['AvailabilityZone']}")
    
    return instance

def manage_ebs_improved(instance):
    az = instance.placement['AvailabilityZone']
    
    print(f"Creando volumen en {az}...")
    volume = ec2_client.create_volume(
        AvailabilityZone=az,
        Size=1,
        VolumeType='gp2'
    )
    vol_id = volume['VolumeId']
    
    waiter = ec2_client.get_waiter('volume_available')
    waiter.wait(VolumeIds=[vol_id])
    
    ec2_client.attach_volume(
        Device='/dev/sdh',
        InstanceId=instance.id,
        VolumeId=vol_id
    )
    print(f"Volumen {vol_id} asociado correctamente.")
    return vol_id

# Ejecución del bloque
mi_instancia = manage_ec2_improved()
mi_volumen_id = manage_ebs_improved(mi_instancia)

Creando instancia...
Instancia i-02599edb706419689 en marcha en la zona us-east-1a
Creando volumen en us-east-1a...
Volumen vol-0b999b5edd431ce85 asociado correctamente.


In [ ]:
# 2. EFS: Elastic File System (Almacenamiento de Archivos)
import time
load_dotenv()
s3_client = boto3.client('s3', region_name=os.getenv('AWS_REGION'))
efs_client = boto3.client('efs', region_name=os.getenv('AWS_REGION'))
ec2_client = boto3.client('ec2', region_name=os.getenv('AWS_REGION'))

def manage_efs_full(subnet_id, security_group_id):
 
    print("Creando el sistema de archivos EFS...")
    fs_response = efs_client.create_file_system(
        CreationToken='ejercicio-ia-bigdata-token',
        PerformanceMode='generalPurpose',
        Encrypted=True,
        Tags=[{'Key': 'Name', 'Value': 'MiEFSCompartido'}]
    )
    fs_id = fs_response['FileSystemId']
    

    print(f"Esperando a que {fs_id} esté disponible...")
    while True:
        status = efs_client.describe_file_systems(FileSystemId=fs_id)['FileSystems'][0]['LifeCycleState']
        if status == 'available': break
        time.sleep(5)


    print(f"Creando punto de montaje en la subred {subnet_id}...")
    efs_client.create_mount_target(
        FileSystemId=fs_id,
        SubnetId=subnet_id,
        SecurityGroups=[security_group_id] # Debe permitir tráfico en puerto 2049
    )

    # 3. Montar y Añadir archivo (Explicación)
    # Para "añadir un archivo", el comando en la EC2 sería:
    # sudo mount -t nfs4 -o nfsvers=4.1,rsize=1048576,wsize=1048576,hard,timeo=600,retrans=2,noresvport fs-id.efs.region.amazonaws.com:/ /mnt/efs
    # echo "Archivo en almacenamiento compartido" > /mnt/efs/datos_compartidos.txt
    
    print(f"EFS listo para ser usado. ID: {fs_id}")
    return fs_id

manage_efs_full(subnet_id='subnet-id', security_group_id='security-group-id')

Creando el sistema de archivos EFS...


FileSystemAlreadyExists: An error occurred (FileSystemAlreadyExists) when calling the CreateFileSystem operation: File system 'fs-04b9b179442fb66d7' already exists with creation token 'ejercicio-ia-bigdata-token'

In [36]:
#3. S3: Almacenamiento de Objetos y Clases de Almacenamiento
import boto3
import os

# Inicialización (asegúrate de que la región esté definida)
region = os.getenv('AWS_REGION')
s3 = boto3.client('s3', region)

def manage_s3_full(bucket_name):
    # --- CREACIÓN DEL BUCKET ---
    # Obligatorio: Bucket (Nombre único global)
    # Opcional: CreateBucketConfiguration (Necesario fuera de us-east-1)
    try:
        # El "truco" está aquí:
        if region == 'us-east-1':
            # us-east-1 NO acepta LocationConstraint
            s3.create_bucket(Bucket=bucket_name)
        else:
            # El resto de regiones LO REQUIEREN obligatoriamente
            s3.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={'LocationConstraint': region}
            )
        print(f"✅ Bucket '{bucket_name}' creado con éxito en {region}")
        
    except s3.exceptions.BucketAlreadyOwnedByYou:
        print(f"⚠️ El bucket '{bucket_name}' ya es tuyo.")
    except Exception as e:
        print(f"❌ Error al crear el bucket: {e}")
        print(f"Bucket {bucket_name} creado.")
    except s3.exceptions.BucketAlreadyOwnedByYou:
        print(f"Ya eres el dueño del bucket {bucket_name}.")

    # --- SUBIDA CON ESTRUCTURA DE CARPETAS Y CLASES ---
    # Almacenaríamos:
    # - STANDARD: Datos de procesamiento inmediato (Data Lakes activos).
    # - IA/GLACIER: Backups, logs históricos o datos de cumplimiento legal.
    
    classes = [
        ('STANDARD', 'ventas/recientes/datos.csv'),
        ('STANDARD_IA', 'backups/mensual/historia.csv'),
        ('INTELLIGENT_TIERING', 'analisis/variable/data.csv'),
        ('GLACIER', 'archivo/2023/reporte.csv'),
        ('DEEP_ARCHIVE', 'archivo/legal/antiguo.csv')
    ]

    csv_body = "id,fecha,monto\n1,2026-03-13,150.50\n2,2026-03-14,89.00"

    for s_class, path in classes:
        # Parámetros de put_object:
        # Obligatorios: Bucket, Key
        # Opcionales: Body, StorageClass, ContentType (útil para que el navegador lo abra bien)
        s3.put_object(
            Bucket=bucket_name,
            Key=path,
            Body=csv_body,
            StorageClass=s_class,
            ContentType='text/csv'
        )
        print(f"Subido a carpeta: {path} [Clase: {s_class}]")

    # --- OBTENER OBJETO ---
    # Obligatorios: Bucket, Key
    # Opcionales: VersionId (si el versionado está activo), Range (para descargar solo una parte)
    obj = s3.get_object(Bucket=bucket_name, Key='ventas/recientes/datos.csv')
    data = obj['Body'].read().decode('utf-8')
    print(f"\nContenido recuperado de ventas/recientes/:\n{data}")

    return bucket_name
manage_s3_full(bucket_name='mi-bucket-ejercicio-ia-bigdata')

✅ Bucket 'mi-bucket-ejercicio-ia-bigdata' creado con éxito en us-east-1
Subido a carpeta: ventas/recientes/datos.csv [Clase: STANDARD]
Subido a carpeta: backups/mensual/historia.csv [Clase: STANDARD_IA]
Subido a carpeta: analisis/variable/data.csv [Clase: INTELLIGENT_TIERING]
Subido a carpeta: archivo/2023/reporte.csv [Clase: GLACIER]
Subido a carpeta: archivo/legal/antiguo.csv [Clase: DEEP_ARCHIVE]

Contenido recuperado de ventas/recientes/:
id,fecha,monto
1,2026-03-13,150.50
2,2026-03-14,89.00


'mi-bucket-ejercicio-ia-bigdata'

In [37]:
#4. Control de Versiones en S3
def s3_versioning_demo_completa(bucket_name):
    print("\n--- Iniciando Ejemplo de Versionado ---")
    
    # 1. Habilitar
    s3.put_bucket_versioning(
        Bucket=bucket_name,
        VersioningConfiguration={'Status': 'Enabled'}
    )
    
    key = 'config/modelo_ia.txt'
    
    # 2. Crear Versión 1 (Original)
    s3.put_object(Bucket=bucket_name, Key=key, Body='Pesos del modelo v1: 0.85')
    
    # 3. Crear Versión 2 (Modificado)
    s3.put_object(Bucket=bucket_name, Key=key, Body='Pesos del modelo v2: 0.92 (Optimizado)')
    
    # 4. Listar y Mostrar contenido de ambas
    print(f"Buscando versiones para: {key}...")
    versions = s3.list_object_versions(Bucket=bucket_name, Prefix=key)
    
    for v in versions.get('Versions', []):
        v_id = v['VersionId']
        # Para "mostrar" las versiones, las descargamos usando su VersionId
        response = s3.get_object(Bucket=bucket_name, Key=key, VersionId=v_id)
        contenido = response['Body'].read().decode('utf-8')
        
        print(f" -> ID: {v_id}")
        print(f"    Fecha: {v['LastModified']}")
        print(f"    Contenido: '{contenido}'")
        print(f"    ¿Es la actual?: {v['IsLatest']}")
        print("-" * 30)

s3_versioning_demo_completa('mi-bucket-de-prueba')


--- Iniciando Ejemplo de Versionado ---


AccessDenied: An error occurred (AccessDenied) when calling the PutBucketVersioning operation: Access Denied

In [39]:
import boto3
import os

athena = boto3.client('athena', region_name=os.getenv('AWS_REGION'))

def setup_athena(bucket_name):
    # --- PARÁMETROS DE ATHENA ---
    # Obligatorios: QueryString, ResultConfiguration (S3 para guardar el resultado)
    # Opcionales: QueryExecutionContext (Database), WorkGroup
    
    output_location = f"s3://{bucket_name}/athena-results/"
    
    # 1. Crear Base de Datos
    athena.start_query_execution(
        QueryString="CREATE DATABASE IF NOT EXISTS ejercicio_db",
        ResultConfiguration={'OutputLocation': output_location}
    )
    
    # 2. Crear Tabla CSV
    create_table_csv = f"""
    CREATE EXTERNAL TABLE IF NOT EXISTS ejercicio_db.ventas_csv (
        id int,
        nombre string,
        valor int
    )
    ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
    LOCATION 's3://{bucket_name}/'
    TBLPROPERTIES ('skip.header.line.count'='1');
    """
    
    athena.start_query_execution(
        QueryString=create_table_csv,
        ResultConfiguration={'OutputLocation': output_location}
    )
    print("Base de datos y tabla CSV creadas en Athena.")

# --- EJEMPLO DE LAS 3 QUERIES SOBRE EL CSV ---
# Q1: "SELECT * FROM ventas_csv WHERE valor > 150;"
# Q2: "SELECT SUM(valor) FROM ventas_csv;"
# Q3: "SELECT COUNT(*) FROM ventas_csv;"
setup_athena('mi-bucket-ejercicio-ia-bigdata')

Base de datos y tabla CSV creadas en Athena.
